# Bài 6 · Cây quyết định · Thực hành trên lớp (v3)

Mở notebook từ thư mục bài. Dữ liệu: `data/olist_model_table.csv` (giống Bài 4–5 v3). Lớp dương: đơn trễ. Mã cho sẵn; việc của bạn là chạy từng ô, đối chiếu với slide và trả lời câu hỏi in đậm.
Quy tắc: chỉ học trên train, chọn độ sâu và ngưỡng trên validation, test chấm một lần.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import average_precision_score
ROOT = Path.cwd()
FEAT = ['distance_km', 'promised_days', 'price', 'freight_value', 'n_items', 'weight_g']
d = pd.read_csv(ROOT / 'data' / 'olist_model_table.csv')
tr, va, te = (d[d.split == k].reset_index(drop=True) for k in ['train', 'validation', 'test'])
imp = SimpleImputer(strategy='median').fit(tr[FEAT])
RAW = {k: imp.transform(a[FEAT]) for k, a in [('train', tr), ('validation', va), ('test', te)]}
y = {'train': tr.late.to_numpy(), 'validation': va.late.to_numpy(), 'test': te.late.to_numpy()}
print({k: (len(v), int(v.sum())) for k, v in y.items()})

{'train': (29031, 941), 'validation': (10389, 634), 'test': (40266, 3478)}


## Chặng 1 · Đọc một cây thật (slide 11)

Cây chỉ dùng khoảng cách và số ngày hứa, sâu 2, mỗi lá ≥ 200 đơn. **Câu hỏi:** ngưỡng “hứa ngắn” ở nhánh gần và nhánh xa là bao nhiêu? Vùng nào trễ nhiều nhất?

In [2]:
cay2 = DecisionTreeClassifier(max_depth=2, min_samples_leaf=200, random_state=42).fit(RAW['train'][:, :2], y['train'])
print(export_text(cay2, feature_names=['khoang_cach_km', 'so_ngay_hua'], show_weights=True, decimals=1))
la = cay2.apply(RAW['train'][:, :2])
print(pd.DataFrame({'la': la, 'tre': y['train']}).groupby('la').tre.agg(['size', 'sum', 'mean']).round(4))

|--- khoang_cach_km <= 1395.3
|   |--- so_ngay_hua <= 9.5
|   |   |--- weights: [264.0, 33.0] class: 0
|   |--- so_ngay_hua >  9.5
|   |   |--- weights: [24825.0, 695.0] class: 0
|--- khoang_cach_km >  1395.3
|   |--- so_ngay_hua <= 26.5
|   |   |--- weights: [312.0, 48.0] class: 0
|   |--- so_ngay_hua >  26.5
|   |   |--- weights: [2689.0, 165.0] class: 0

     size  sum    mean
la                    
2     297   33  0.1111
3   25520  695  0.0272
5     360   48  0.1333
6    2854  165  0.0578


In [3]:
# TỰ KIỂM · chặng 1
t = cay2.tree_
assert abs(t.threshold[0] - 1395.2736) < 1e-2, 'gốc hỏi khoảng cách ≤ 1.395 km'
print('✅ gốc: khoảng cách ≤', round(t.threshold[0], 1), 'km; nhánh gần hứa ≤', round(t.threshold[t.children_left[0]], 1), '; nhánh xa hứa ≤', round(t.threshold[t.children_right[0]], 1))

✅ gốc: khoảng cách ≤ 1395.3 km; nhánh gần hứa ≤ 9.5 ; nhánh xa hứa ≤ 26.5


## Chặng 2 · Gini, và vì sao không dùng tỷ lệ lỗi (slide 18)

Hàm dưới quét mọi ngưỡng của một cột bằng tổng tích luỹ và trả về Gini sau chia và tỷ lệ lỗi sau chia cho từng ứng viên (mỗi con ≥ 200 đơn).

In [4]:
def gini(n, k):
    p = k / n
    return 1 - p * p - (1 - p) * (1 - p)

def quet(x, yy, m=200):
    n, k = len(yy), yy.sum(); o = np.argsort(x, kind='stable'); xs, ys = x[o], yy[o]
    nl = np.arange(1, n); kl = np.cumsum(ys)[:-1]
    ok = (xs[1:] > xs[:-1] + 1e-7) & (nl >= m) & (n - nl >= m)
    nl, kl = nl[ok], kl[ok]; nr, kr = n - nl, k - kl
    g = (nl * gini(nl, kl) + nr * gini(nr, kr)) / n
    loi = (np.minimum(kl, nl - kl) + np.minimum(kr, nr - kr)) / n
    nguong = (xs[np.where(ok)[0]] + xs[np.where(ok)[0] + 1]) / 2
    return nguong, g, loi

g0 = gini(len(y['train']), y['train'].sum())
for j, f in enumerate(FEAT):
    t_, g, loi = quet(RAW['train'][:, j], y['train'])
    print(f'{f:15s} ứng viên {len(t_):6d} · tốt nhất ≤ {t_[np.argmin(g)]:10.2f} · Δ = {g0 - g.min():.6f} · tỷ lệ lỗi sau chia: min {loi.min():.4f}, max {loi.max():.4f}')

distance_km     ứng viên  27464 · tốt nhất ≤    1395.27 · Δ = 0.000285 · tỷ lệ lỗi sau chia: min 0.0324, max 0.0324
promised_days   ứng viên  27028 · tốt nhất ≤       9.54 · Δ = 0.000128 · tỷ lệ lỗi sau chia: min 0.0324, max 0.0324
price           ứng viên   3384 · tốt nhất ≤     207.42 · Δ = 0.000034 · tỷ lệ lỗi sau chia: min 0.0324, max 0.0324
freight_value   ứng viên   4022 · tốt nhất ≤      18.91 · Δ = 0.000113 · tỷ lệ lỗi sau chia: min 0.0324, max 0.0324
n_items         ứng viên      3 · tốt nhất ≤       2.50 · Δ = 0.000003 · tỷ lệ lỗi sau chia: min 0.0324, max 0.0324
weight_g        ứng viên   1501 · tốt nhất ≤     253.50 · Δ = 0.000035 · tỷ lệ lỗi sau chia: min 0.0324, max 0.0324


In [5]:
# TỰ KIỂM · chặng 2
t_, g, loi = quet(RAW['train'][:, 0], y['train'])
assert np.allclose(loi, y['train'].mean()), 'mọi nhát cắt giữ tỷ lệ lỗi bằng tỷ lệ trễ train'
assert abs(g0 - 0.0627259672) < 1e-9
print('✅ G gốc =', round(g0, 5), '; tỷ lệ lỗi đứng yên ở', round(loi[0], 4), 'với mọi ngưỡng khoảng cách')

✅ G gốc = 0.06273 ; tỷ lệ lỗi đứng yên ở 0.0324 với mọi ngưỡng khoảng cách


**Câu hỏi:** cột nào có Δ lớn nhất ở gốc? Δ nhỏ cỡ nào so với G gốc, và vì sao?

## Chặng 3 · Thuật toán và scikit-learn (slide 42–43)

Kết quả quét ở trên phải trùng nút gốc của scikit-learn. Sau đó dựng cây sâu 6 và xem 3 tầng đầu.

In [6]:
sc = StandardScaler().fit(RAW['train'])
X = {k: sc.transform(v) for k, v in RAW.items()}
cay6 = DecisionTreeClassifier(max_depth=6, min_samples_leaf=200, random_state=42).fit(X['train'], y['train'])
goc_km = cay6.tree_.threshold[0] * sc.scale_[0] + sc.mean_[0]
print('scikit-learn: gốc hỏi cột', FEAT[cay6.tree_.feature[0]], '≤', round(goc_km, 2), 'km;', cay6.tree_.node_count, 'nút,', cay6.get_n_leaves(), 'lá')
print(export_text(cay6, feature_names=FEAT, max_depth=3, decimals=2))

scikit-learn: gốc hỏi cột distance_km ≤ 1395.27 km; 45 nút, 23 lá
|--- distance_km <= 1.29
|   |--- promised_days <= -1.85
|   |   |--- class: 0
|   |--- promised_days >  -1.85
|   |   |--- promised_days <= -0.80
|   |   |   |--- weight_g <= -0.27
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- weight_g >  -0.27
|   |   |   |   |--- truncated branch of depth 3
|   |   |--- promised_days >  -0.80
|   |   |   |--- weight_g <= 3.74
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- weight_g >  3.74
|   |   |   |   |--- class: 0
|--- distance_km >  1.29
|   |--- promised_days <= 0.29
|   |   |--- class: 0
|   |--- promised_days >  0.29
|   |   |--- promised_days <= 1.29
|   |   |   |--- freight_value <= -0.09
|   |   |   |   |--- class: 0
|   |   |   |--- freight_value >  -0.09
|   |   |   |   |--- truncated branch of depth 3
|   |   |--- promised_days >  1.29
|   |   |   |--- price <= 0.25
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- p

In [7]:
# TỰ KIỂM · chặng 3
assert cay6.tree_.feature[0] == 0 and abs(goc_km - 1395.2736) < 1e-2 and cay6.get_n_leaves() == 23
print('✅ nút gốc trùng phép quét tự viết; cây có 23 lá')

✅ nút gốc trùng phép quét tự viết; cây có 23 lá


## Chặng 4 · Độ phức tạp, chọn trên validation, test một lần (slide 47–55)

In [8]:
for msl in [1, 200]:
    for dd in [2, 4, 6, 8, None]:
        q = DecisionTreeClassifier(max_depth=dd, min_samples_leaf=msl, random_state=42).fit(X['train'], y['train'])
        print(f'lá ≥ {msl:3d} · sâu {str(dd):4s} · {q.get_n_leaves():5d} lá · AP train {average_precision_score(y["train"], q.predict_proba(X["train"])[:, 1]):.4f} · AP validation {average_precision_score(y["validation"], q.predict_proba(X["validation"])[:, 1]):.4f}')

lá ≥   1 · sâu 2    ·     4 lá · AP train 0.0475 · AP validation 0.0773


lá ≥   1 · sâu 4    ·    14 lá · AP train 0.0713 · AP validation 0.0767


lá ≥   1 · sâu 6    ·    37 lá · AP train 0.1009 · AP validation 0.0778


lá ≥   1 · sâu 8    ·    66 lá · AP train 0.1285 · AP validation 0.0853


lá ≥   1 · sâu None ·  1640 lá · AP train 1.0000 · AP validation 0.0615
lá ≥ 200 · sâu 2    ·     4 lá · AP train 0.0474 · AP validation 0.0782
lá ≥ 200 · sâu 4    ·    10 lá · AP train 0.0574 · AP validation 0.0860


lá ≥ 200 · sâu 6    ·    23 lá · AP train 0.0667 · AP validation 0.1005
lá ≥ 200 · sâu 8    ·    41 lá · AP train 0.0728 · AP validation 0.1021


lá ≥ 200 · sâu None ·   108 lá · AP train 0.0831 · AP validation 0.0993


**Câu hỏi:** cây tự do (lá ≥ 1, không giới hạn sâu) có AP validation so với mốc hằng số (tỷ lệ trễ validation) thế nào?

In [9]:
# Lưới khoá {3; 6}, lá ≥ 200: chọn theo AP validation; rồi chọn τ theo FP + 5·FN trên validation
ung = {dd: DecisionTreeClassifier(max_depth=dd, min_samples_leaf=200, random_state=42).fit(X['train'], y['train']) for dd in [3, 6]}
ap = {dd: average_precision_score(y['validation'], m.predict_proba(X['validation'])[:, 1]) for dd, m in ung.items()}
DD = max(ap, key=lambda dd: (ap[dd], -ung[dd].get_n_leaves())); M = ung[DD]
pv = M.predict_proba(X['validation'])[:, 1]
def chi_phi(yy, p, t):
    c = p >= t
    return int((c & (yy == 0)).sum()) + 5 * int((~c & (yy == 1)).sum()), int(c.sum())
bang = {t: chi_phi(y['validation'], pv, t) for t in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]}
TAU = min(bang, key=lambda t: (bang[t][0], bang[t][1], -t))
print('AP validation', {k: round(v, 4) for k, v in ap.items()}, '→ sâu', DD); print('chi phí validation', bang)
print('τ khoá =', TAU)

AP validation {3: 0.0817, 6: 0.1005} → sâu 6
chi phí validation {0.05: (3395, 975), 0.1: (3132, 352), 0.15: (3170, 0), 0.2: (3170, 0), 0.3: (3170, 0), 0.5: (3170, 0)}
τ khoá = 0.1


In [10]:
# TEST · chạy đúng một lần sau khi đã khoá sâu và τ
pt = M.predict_proba(X['test'])[:, 1]; c = pt >= TAU; yt = y['test']
tn, fp, fn, tp = int((~c & (yt == 0)).sum()), int((c & (yt == 0)).sum()), int((~c & (yt == 1)).sum()), int((c & (yt == 1)).sum())
print('test · TN, FP, FN, TP =', (tn, fp, fn, tp), '· chi phí', fp + 5 * fn, '· không cảnh báo', 5 * yt.sum(), '· logistic Bài 5: 16944')
assert (tn, fp, fn, tp) == (34659, 2129, 3046, 432)
print('✅ khớp slide 55')

test · TN, FP, FN, TP = (34659, 2129, 3046, 432) · chi phí 17359 · không cảnh báo 17390 · logistic Bài 5: 16944
✅ khớp slide 55


**Câu hỏi cuối:** trên cùng validation, AP của cây so với logistic Bài 5 (0.1165)? Ở vùng “xa · hứa ngắn”, tỷ lệ trễ thật gấp mấy lần vùng “xa · hứa dài”, và mỗi mô hình cho mức chênh bao nhiêu (cột `region`, `p_tree`, `p_logit` trong `data/olist_validation_tree_scores_v3.csv`)? Viết 3–4 câu, có số.